# PatchCore on MVTec AD capsule

In [1]:
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score
from torchvision import models, transforms
from tqdm import tqdm

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

DATA_DIR = Path("capsule")
IMG_SIZE = 320
CORESET_RATIO = 0.1
BATCH_SIZE = 16
HOLDOUT_RATIO = 0.25
TARGET_FPR = 0.02
SEED = 0
torch.manual_seed(SEED)
print(device)

mps


In [2]:
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE * 256 // 224),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_images(paths):
    return torch.stack([transform(Image.open(p).convert("RGB")) for p in paths])


all_train_paths = sorted((DATA_DIR / "train" / "good").glob("*.png"))
perm = np.random.default_rng(SEED).permutation(len(all_train_paths))
n_holdout = int(len(all_train_paths) * HOLDOUT_RATIO)
calib_paths = [all_train_paths[i] for i in perm[:n_holdout]]
train_paths = [all_train_paths[i] for i in perm[n_holdout:]]

test_paths, test_labels = [], []
for defect_dir in sorted((DATA_DIR / "test").iterdir()):
    for p in sorted(defect_dir.glob("*.png")):
        test_paths.append(p)
        test_labels.append(0 if defect_dir.name == "good" else 1)
test_labels = np.array(test_labels)

print(f"memory bank: {len(train_paths)} | calibration: {len(calib_paths)}")
print(f"test: {len(test_paths)} (good: {(test_labels == 0).sum()}, defective: {test_labels.sum()})")

memory bank: 165 | calibration: 54
test: 132 (good: 23, defective: 109)


In [3]:
backbone = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(device).eval()


@torch.no_grad()
def embed(images):
    x = images.to(device)
    x = backbone.maxpool(backbone.relu(backbone.bn1(backbone.conv1(x))))
    x = backbone.layer1(x)
    f2 = backbone.layer2(x)   # [B, 512, H/8, W/8]
    f3 = backbone.layer3(f2)  # [B, 1024, H/16, W/16]

    # Average each position with its 3x3 neighbours; stride=1 + padding=1 keeps the spatial size unchanged
    f2 = F.avg_pool2d(f2, kernel_size=3, stride=1, padding=1)
    f3 = F.avg_pool2d(f3, kernel_size=3, stride=1, padding=1)
    f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)

    f = torch.cat([f2, f3], dim=1)  # [B, 1536, H/8, W/8]
    # Move channels last, then flatten the spatial grid: one 1536-d vector per patch -> [B, n_patches, 1536]
    return f.permute(0, 2, 3, 1).reshape(f.shape[0], -1, f.shape[1])

In [4]:
train_feats = []
for i in tqdm(range(0, len(train_paths), BATCH_SIZE), desc="train features"):
    feats = embed(load_images(train_paths[i:i + BATCH_SIZE]))
    train_feats.append(feats.reshape(-1, feats.shape[-1]).cpu())
train_feats = torch.cat(train_feats)
print(train_feats.shape)

train features: 100%|██████████| 11/11 [00:07<00:00,  1.46it/s]


torch.Size([264000, 1536])


In [5]:
def greedy_coreset(feats, ratio, proj_dim=128):
    # Random projection to 128-d: distances stay roughly the same, but are much cheaper to compute
    proj = torch.randn(feats.shape[1], proj_dim) / proj_dim ** 0.5
    z = (feats @ proj).to(device)

    n_select = int(len(z) * ratio)
    selected = [torch.randint(len(z), (1,)).item()]
    # min_dist[j] = distance from point j to its nearest already-selected point
    min_dist = torch.linalg.norm(z - z[selected[0]], dim=1)

    for _ in tqdm(range(n_select - 1), desc="coreset"):
        idx = torch.argmax(min_dist).item()  # the point worst covered by the coreset so far
        selected.append(idx)
        min_dist = torch.minimum(min_dist, torch.linalg.norm(z - z[idx], dim=1))

    return feats[selected]  # original 1536-d features, not the projected ones


memory_bank = greedy_coreset(train_feats, CORESET_RATIO).to(device)
del train_feats
print(memory_bank.shape)

coreset: 100%|██████████| 26399/26399 [06:16<00:00, 70.08it/s]


torch.Size([26400, 1536])


In [6]:
def image_scores(paths, desc):
    scores = []
    for i in tqdm(range(0, len(paths), BATCH_SIZE), desc=desc):
        feats = embed(load_images(paths[i:i + BATCH_SIZE]))
        for patches in feats:
            # cdist: all pairwise distances [n_patches, M]; min over the bank = patch score, max over patches = image score
            patch_scores = torch.cdist(patches, memory_bank).min(dim=1).values
            scores.append(patch_scores.max().item())
    return np.array(scores)


calib_scores = image_scores(calib_paths, "calibration scores")
test_scores = image_scores(test_paths, "test scores")

test scores: 100%|██████████| 9/9 [00:18<00:00,  2.07s/it]


In [7]:
# Threshold from held-out good images only: allow at most k of them above it, where k / n <= TARGET_FPR.
# Sorted descending, the (k+1)-th highest score is the threshold; anything strictly above it is flagged defective.
k = int(np.floor(TARGET_FPR * len(calib_scores)))
threshold = np.sort(calib_scores)[::-1][k]

test_pred = test_scores > threshold
calib_fpr = (calib_scores > threshold).mean()
test_recall = test_pred[test_labels == 1].mean()
test_fpr = test_pred[test_labels == 0].mean()

print(f"Image AUROC (test):  {roc_auc_score(test_labels, test_scores):.4f}")
print(f"Threshold:           {threshold:.4f}")
print(f"Calibration FPR:     {calib_fpr:.4f}  ({(calib_scores > threshold).sum()}/{len(calib_scores)} held-out good images flagged)")
print(f"Test recall:         {test_recall:.4f}  ({test_pred[test_labels == 1].sum()}/{(test_labels == 1).sum()} defects caught)")
print(f"Test FPR:            {test_fpr:.4f}  ({test_pred[test_labels == 0].sum()}/{(test_labels == 0).sum()} good images flagged)")

Image AUROC (test):  0.9836
Threshold:           1.8648
Calibration FPR:     0.0185  (1/54 held-out good images flagged)
Test recall:         0.8624  (94/109 defects caught)
Test FPR:            0.0435  (1/23 good images flagged)
